In [ ]:
#!/usr/bin/env python3
"""
Multimodal biometrics demo:
- Face: FaceNet embeddings (facenet-pytorch) on LFW or folder dataset
- Fingerprint: minutiae matching + CNN embedding
- Palm: CNN embedding + Gabor feature
- Fusion: weighted score fusion + logistic regression score fusion

This script is intentionally end-to-end and self-contained for a project/demo setting.
"""

from __future__ import annotations

import argparse
import itertools
import json
import math
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from sklearn.datasets import fetch_lfw_people
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import train_test_split
from skimage.filters import gabor
from skimage.morphology import skeletonize
from skimage.transform import resize
from tqdm import tqdm

try:
    import timm
except ImportError as e:
    raise ImportError("Please install timm: pip install timm") from e

try:
    from facenet_pytorch import InceptionResnetV1
except ImportError:
    InceptionResnetV1 = None


# -----------------------------
# Utilities
# -----------------------------

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@dataclass
class Sample:
    identity: str
    path: Optional[str]
    image: np.ndarray  # HxW or HxWx3, uint8


@dataclass
class PairRecord:
    a_idx: int
    b_idx: int
    label: int
    face_score: float
    fp_score: float
    palm_score: float


@dataclass
class EvalMetrics:
    eer: float
    eer_threshold: float
    far_at_eer: float
    frr_at_eer: float
    auc: float


# -----------------------------
# Loading data
# -----------------------------

def load_folder_dataset(root: str, color: bool = False, max_identities: Optional[int] = None,
                        max_images_per_identity: Optional[int] = None) -> List[Sample]:
    root_path = Path(root)
    if not root_path.exists():
        raise FileNotFoundError(f"Dataset folder does not exist: {root}")

    samples: List[Sample] = []
    identity_dirs = sorted([p for p in root_path.iterdir() if p.is_dir()])
    if max_identities is not None:
        identity_dirs = identity_dirs[:max_identities]

    for identity_dir in identity_dirs:
        img_paths = sorted([p for p in identity_dir.rglob("*") if p.suffix.lower() in IMG_EXTS])
        if max_images_per_identity is not None:
            img_paths = img_paths[:max_images_per_identity]
        for img_path in img_paths:
            if color:
                img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
                if img is None:
                    continue
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            else:
                img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
                if img is None:
                    continue
            samples.append(Sample(identity=identity_dir.name, path=str(img_path), image=img))
    return samples


def load_lfw_dataset(min_faces_per_person: int = 5, resize_factor: float = 0.5,
                     max_identities: Optional[int] = None,
                     max_images_per_identity: Optional[int] = None) -> List[Sample]:
    data = fetch_lfw_people(min_faces_per_person=min_faces_per_person, resize=resize_factor, color=True)
    X = data.images  # float64, 0..255
    y = data.target
    target_names = data.target_names

    samples: List[Sample] = []
    selected_ids = list(range(len(target_names)))
    if max_identities is not None:
        selected_ids = selected_ids[:max_identities]

    per_id_counter: Dict[int, int] = {i: 0 for i in selected_ids}
    for img, target in zip(X, y):
        if target not in selected_ids:
            continue
        if max_images_per_identity is not None and per_id_counter[target] >= max_images_per_identity:
            continue
        per_id_counter[target] += 1
        img_uint8 = np.clip(img, 0, 255).astype(np.uint8)
        samples.append(Sample(identity=str(target_names[target]), path=None, image=img_uint8))
    return samples


# -----------------------------
# Preprocessing
# -----------------------------

def ensure_uint8(img: np.ndarray) -> np.ndarray:
    if img.dtype == np.uint8:
        return img
    img = np.asarray(img)
    img = img.astype(np.float32)
    img -= img.min()
    denom = max(img.max(), 1e-6)
    img = (255.0 * img / denom).clip(0, 255).astype(np.uint8)
    return img


def to_gray(img: np.ndarray) -> np.ndarray:
    if img.ndim == 2:
        return img
    return cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)


def preprocess_fingerprint(img: np.ndarray, out_size: Tuple[int, int] = (224, 224)) -> np.ndarray:
    gray = to_gray(ensure_uint8(img))
    blur = cv2.GaussianBlur(gray, (3, 3), 0)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(blur)
    norm = cv2.normalize(clahe, None, 0, 255, cv2.NORM_MINMAX)
    out = cv2.resize(norm, out_size, interpolation=cv2.INTER_CUBIC)
    return out


def preprocess_palm(img: np.ndarray, out_size: Tuple[int, int] = (224, 224)) -> np.ndarray:
    gray = to_gray(ensure_uint8(img))
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(blur)
    out = cv2.resize(clahe, out_size, interpolation=cv2.INTER_CUBIC)
    return out


def preprocess_face(img: np.ndarray, out_size: Tuple[int, int] = (160, 160)) -> np.ndarray:
    img = ensure_uint8(img)
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    out = cv2.resize(img, out_size, interpolation=cv2.INTER_CUBIC)
    return out


# -----------------------------
# Feature extractors
# -----------------------------

class CNNEmbeddingExtractor:
    def __init__(self, model_name: str = "resnet18", device: str = "cpu"):
        self.device = torch.device(device)
        self.model = timm.create_model(model_name, pretrained=True, num_classes=0, global_pool="avg")
        self.model.eval().to(self.device)
        data_cfg = timm.data.resolve_model_data_config(self.model)
        self.input_size = data_cfg.get("input_size", (3, 224, 224))
        self.mean = torch.tensor(data_cfg.get("mean", (0.485, 0.456, 0.406)), device=self.device).view(1, 3, 1, 1)
        self.std = torch.tensor(data_cfg.get("std", (0.229, 0.224, 0.225)), device=self.device).view(1, 3, 1, 1)

    def _prepare(self, img_gray_or_rgb: np.ndarray) -> torch.Tensor:
        img = ensure_uint8(img_gray_or_rgb)
        h, w = self.input_size[1], self.input_size[2]
        if img.ndim == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        img = cv2.resize(img, (w, h), interpolation=cv2.INTER_CUBIC)
        x = torch.from_numpy(img).permute(2, 0, 1).float().unsqueeze(0).to(self.device) / 255.0
        x = (x - self.mean) / self.std
        return x

    @torch.inference_mode()
    def extract(self, img: np.ndarray) -> np.ndarray:
        x = self._prepare(img)
        emb = self.model(x)
        emb = F.normalize(emb, dim=1)
        return emb.squeeze(0).cpu().numpy().astype(np.float32)


class FaceNetExtractor:
    def __init__(self, device: str = "cpu"):
        if InceptionResnetV1 is None:
            raise ImportError("Please install facenet-pytorch")
        self.device = torch.device(device)
        self.model = InceptionResnetV1(pretrained="vggface2").eval().to(self.device)

    @torch.inference_mode()
    def extract(self, img: np.ndarray) -> np.ndarray:
        img = preprocess_face(img, (160, 160))
        x = torch.from_numpy(img).permute(2, 0, 1).float().unsqueeze(0).to(self.device)
        x = (x / 255.0 - 0.5) / 0.5
        emb = self.model(x)
        emb = F.normalize(emb, dim=1)
        return emb.squeeze(0).cpu().numpy().astype(np.float32)


# -----------------------------
# Fingerprint minutiae
# -----------------------------

def fingerprint_binarize_and_skeletonize(gray: np.ndarray) -> np.ndarray:
    gray = preprocess_fingerprint(gray)
    # adaptive threshold + thinning-ready binary image
    binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY_INV, 15, 5)
    binary = cv2.medianBlur(binary, 3)
    skel = skeletonize(binary > 0)
    return skel.astype(np.uint8)


def crossing_number(neighborhood: np.ndarray) -> int:
    vals = neighborhood.flatten().tolist()
    # P2 P3 P4 P5 P6 P7 P8 P9 order around center
    p = [vals[1], vals[2], vals[5], vals[8], vals[7], vals[6], vals[3], vals[0], vals[1]]
    return int(sum(abs(p[i] - p[i + 1]) for i in range(8)) / 2)


def estimate_orientation(skel: np.ndarray, y: int, x: int, win: int = 5) -> float:
    y0, y1 = max(0, y - win), min(skel.shape[0], y + win + 1)
    x0, x1 = max(0, x - win), min(skel.shape[1], x + win + 1)
    pts = np.argwhere(skel[y0:y1, x0:x1] > 0)
    if len(pts) < 2:
        return 0.0
    pts = pts.astype(np.float32)
    pts[:, 0] += y0
    pts[:, 1] += x0
    pts -= pts.mean(axis=0, keepdims=True)
    cov = np.cov(pts.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    v = eigvecs[:, np.argmax(eigvals)]
    return float(math.atan2(v[0], v[1]))


def extract_minutiae(gray: np.ndarray, max_points: int = 120) -> np.ndarray:
    skel = fingerprint_binarize_and_skeletonize(gray)
    points: List[Tuple[float, float, float, float]] = []
    for y in range(1, skel.shape[0] - 1):
        for x in range(1, skel.shape[1] - 1):
            if skel[y, x] == 0:
                continue
            patch = skel[y - 1:y + 2, x - 1:x + 2]
            cn = crossing_number(patch)
            if cn in (1, 3):  # ridge ending or bifurcation
                ori = estimate_orientation(skel, y, x)
                mtype = 0.0 if cn == 1 else 1.0
                points.append((float(x), float(y), ori, mtype))

    if not points:
        return np.zeros((0, 4), dtype=np.float32)

    pts = np.array(points, dtype=np.float32)
    # Keep strongest spatially-dispersed subset using simple stride sampling after sorting by distance to centroid
    centroid = pts[:, :2].mean(axis=0, keepdims=True)
    d = np.linalg.norm(pts[:, :2] - centroid, axis=1)
    order = np.argsort(-d)
    pts = pts[order]
    pts = pts[:max_points]
    return pts


def minutiae_match_score(m1: np.ndarray, m2: np.ndarray,
                         spatial_thresh: float = 18.0,
                         angle_thresh: float = 0.45) -> float:
    if len(m1) == 0 or len(m2) == 0:
        return 0.0

    p1 = m1[:, :2].copy()
    p2 = m2[:, :2].copy()

    p1 -= p1.mean(axis=0, keepdims=True)
    p2 -= p2.mean(axis=0, keepdims=True)

    D = cdist(p1, p2)
    row_ind, col_ind = linear_sum_assignment(D)
    matches = 0
    type_bonus = 0
    for i, j in zip(row_ind, col_ind):
        if D[i, j] > spatial_thresh:
            continue
        a1 = m1[i, 2]
        a2 = m2[j, 2]
        da = abs(math.atan2(math.sin(a1 - a2), math.cos(a1 - a2)))
        if da <= angle_thresh:
            matches += 1
            if m1[i, 3] == m2[j, 3]:
                type_bonus += 1
    denom = max(len(m1), len(m2), 1)
    return float((matches + 0.25 * type_bonus) / denom)


# -----------------------------
# Palm Gabor
# -----------------------------

def extract_gabor_features(gray: np.ndarray,
                           frequencies: Sequence[float] = (0.1, 0.2, 0.3),
                           thetas: Sequence[float] = (0, np.pi/4, np.pi/2, 3*np.pi/4)) -> np.ndarray:
    gray = preprocess_palm(gray)
    gray = gray.astype(np.float32) / 255.0
    feats: List[float] = []
    for freq in frequencies:
        for theta in thetas:
            real, imag = gabor(gray, frequency=freq, theta=theta)
            mag = np.sqrt(real ** 2 + imag ** 2)
            feats.extend([mag.mean(), mag.std(), np.percentile(mag, 25), np.percentile(mag, 75)])
    feat = np.asarray(feats, dtype=np.float32)
    feat /= (np.linalg.norm(feat) + 1e-8)
    return feat


# -----------------------------
# Similarity scoring
# -----------------------------

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denom = (np.linalg.norm(a) * np.linalg.norm(b)) + 1e-8
    return float(np.dot(a, b) / denom)


def normalize_scores(train_scores: np.ndarray, test_scores: np.ndarray) -> np.ndarray:
    lo = float(np.min(train_scores))
    hi = float(np.max(train_scores))
    if hi - lo < 1e-8:
        return np.zeros_like(test_scores)
    return (test_scores - lo) / (hi - lo)


def compute_eer(y_true: np.ndarray, scores: np.ndarray) -> EvalMetrics:
    fpr, tpr, thresholds = roc_curve(y_true, scores)
    fnr = 1 - tpr
    idx = int(np.nanargmin(np.abs(fpr - fnr)))
    eer = float((fpr[idx] + fnr[idx]) / 2.0)
    roc_auc = float(auc(fpr, tpr))
    return EvalMetrics(
        eer=eer,
        eer_threshold=float(thresholds[idx]),
        far_at_eer=float(fpr[idx]),
        frr_at_eer=float(fnr[idx]),
        auc=roc_auc,
    )


def plot_roc(y_true: np.ndarray, scores: np.ndarray, title: str, out_path: str) -> None:
    fpr, tpr, _ = roc_curve(y_true, scores)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC={roc_auc:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=160)
    plt.close()


# -----------------------------
# Pair building
# -----------------------------

def group_by_identity(samples: List[Sample]) -> Dict[str, List[int]]:
    grouped: Dict[str, List[int]] = {}
    for idx, s in enumerate(samples):
        grouped.setdefault(s.identity, []).append(idx)
    return grouped


def build_pair_indices(samples: List[Sample], max_positive_per_id: int = 10,
                       max_negative: int = 2000) -> List[Tuple[int, int, int]]:
    grouped = group_by_identity(samples)
    pairs: List[Tuple[int, int, int]] = []
    ids = sorted(grouped.keys())

    # positives
    for identity in ids:
        idxs = grouped[identity]
        pos_pairs = list(itertools.combinations(idxs, 2))
        random.shuffle(pos_pairs)
        for a, b in pos_pairs[:max_positive_per_id]:
            pairs.append((a, b, 1))

    # negatives
    neg_candidates: List[Tuple[int, int]] = []
    for i, id1 in enumerate(ids):
        for id2 in ids[i + 1:]:
            for a in grouped[id1][:2]:
                for b in grouped[id2][:2]:
                    neg_candidates.append((a, b))
    random.shuffle(neg_candidates)
    for a, b in neg_candidates[:max_negative]:
        pairs.append((a, b, 0))

    random.shuffle(pairs)
    return pairs


# -----------------------------
# Modalities
# -----------------------------

def extract_face_embeddings(samples: List[Sample], model_name: str, device: str) -> List[np.ndarray]:
    if model_name == "facenet":
        extractor = FaceNetExtractor(device=device)
    else:
        extractor = CNNEmbeddingExtractor(model_name=model_name, device=device)
    feats = []
    for s in tqdm(samples, desc="Face embeddings"):
        feats.append(extractor.extract(s.image))
    return feats


def extract_fingerprint_features(samples: List[Sample], device: str) -> Tuple[List[np.ndarray], List[np.ndarray]]:
    cnn_extractor = CNNEmbeddingExtractor(model_name="resnet18", device=device)
    minutiae_list: List[np.ndarray] = []
    cnn_list: List[np.ndarray] = []
    for s in tqdm(samples, desc="Fingerprint features"):
        gray = preprocess_fingerprint(s.image)
        minutiae = extract_minutiae(gray)
        cnn = cnn_extractor.extract(gray)
        minutiae_list.append(minutiae)
        cnn_list.append(cnn)
    return minutiae_list, cnn_list


def extract_palm_features(samples: List[Sample], device: str) -> Tuple[List[np.ndarray], List[np.ndarray]]:
    cnn_extractor = CNNEmbeddingExtractor(model_name="resnet18", device=device)
    gabor_list: List[np.ndarray] = []
    cnn_list: List[np.ndarray] = []
    for s in tqdm(samples, desc="Palm features"):
        gray = preprocess_palm(s.image)
        gabor_feat = extract_gabor_features(gray)
        cnn_feat = cnn_extractor.extract(gray)
        gabor_list.append(gabor_feat)
        cnn_list.append(cnn_feat)
    return gabor_list, cnn_list


# -----------------------------
# Fusion scoring
# -----------------------------

def build_score_table(face_samples: List[Sample], face_feats: List[np.ndarray],
                      fp_samples: List[Sample], fp_minutiae: List[np.ndarray], fp_cnn: List[np.ndarray],
                      palm_samples: List[Sample], palm_gabor: List[np.ndarray], palm_cnn: List[np.ndarray]) -> pd.DataFrame:
    common_ids = sorted(set(s.identity for s in face_samples)
                        & set(s.identity for s in fp_samples)
                        & set(s.identity for s in palm_samples))
    if len(common_ids) < 2:
        raise RuntimeError("Need at least 2 common identities across face, fingerprint, and palm datasets.")

    # Pick one-to-one aligned samples by identity order.
    # For a research demo, we use up to the minimum image count available per identity across modalities.
    face_group = group_by_identity(face_samples)
    fp_group = group_by_identity(fp_samples)
    palm_group = group_by_identity(palm_samples)

    aligned: List[Tuple[int, int, int, str]] = []
    for identity in common_ids:
        n = min(len(face_group[identity]), len(fp_group[identity]), len(palm_group[identity]))
        for k in range(n):
            aligned.append((face_group[identity][k], fp_group[identity][k], palm_group[identity][k], identity))

    # Convert aligned samples into a single identity-indexed list for pair scoring.
    records = []
    for i in tqdm(range(len(aligned)), desc="Pair scoring"):
        fi, fpi, pi, id_i = aligned[i]
        for j in range(i + 1, len(aligned)):
            fj, fpj, pj, id_j = aligned[j]
            label = 1 if id_i == id_j else 0

            face_score = cosine_similarity(face_feats[fi], face_feats[fj])
            fp_score = 0.6 * cosine_similarity(fp_cnn[fpi], fp_cnn[fpj]) + 0.4 * minutiae_match_score(fp_minutiae[fpi], fp_minutiae[fpj])
            palm_score = 0.65 * cosine_similarity(palm_cnn[pi], palm_cnn[pj]) + 0.35 * cosine_similarity(palm_gabor[pi], palm_gabor[pj])
            records.append({
                "label": label,
                "face_score": face_score,
                "fp_score": fp_score,
                "palm_score": palm_score,
            })
    return pd.DataFrame.from_records(records)


def train_fusion_models(df: pd.DataFrame, random_state: int = 42) -> Tuple[pd.DataFrame, Dict[str, EvalMetrics], LogisticRegression]:
    X = df[["face_score", "fp_score", "palm_score"]].values.astype(np.float32)
    y = df["label"].values.astype(np.int32)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=random_state
    )

    # Score normalization from train split only
    X_train_n = X_train.copy()
    X_test_n = X_test.copy()
    for c in range(X.shape[1]):
        X_train_n[:, c] = normalize_scores(X_train[:, c], X_train[:, c])
        X_test_n[:, c] = normalize_scores(X_train[:, c], X_test[:, c])

    # Weighted fusion: equal-ish weights, slightly favor face in many public datasets
    weighted_train = 0.4 * X_train_n[:, 0] + 0.3 * X_train_n[:, 1] + 0.3 * X_train_n[:, 2]
    weighted_test = 0.4 * X_test_n[:, 0] + 0.3 * X_test_n[:, 1] + 0.3 * X_test_n[:, 2]

    # Logistic fusion
    clf = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=random_state)
    clf.fit(X_train_n, y_train)
    logistic_test = clf.predict_proba(X_test_n)[:, 1]

    metrics = {
        "face": compute_eer(y_test, X_test_n[:, 0]),
        "fingerprint": compute_eer(y_test, X_test_n[:, 1]),
        "palm": compute_eer(y_test, X_test_n[:, 2]),
        "weighted_fusion": compute_eer(y_test, weighted_test),
        "logistic_fusion": compute_eer(y_test, logistic_test),
    }

    eval_df = pd.DataFrame({
        "label": y_test,
        "face_score": X_test_n[:, 0],
        "fp_score": X_test_n[:, 1],
        "palm_score": X_test_n[:, 2],
        "weighted_fusion": weighted_test,
        "logistic_fusion": logistic_test,
    })
    return eval_df, metrics, clf


# -----------------------------
# Main
# -----------------------------

def check_min_images(samples: List[Sample], modality: str) -> None:
    grouped = group_by_identity(samples)
    valid = {k: v for k, v in grouped.items() if len(v) >= 2}
    if len(valid) < 2:
        raise RuntimeError(f"{modality}: need at least 2 identities with 2+ images each.")


def filter_identities_with_min_samples(samples: List[Sample], min_samples: int = 2) -> List[Sample]:
    grouped = group_by_identity(samples)
    keep = {k for k, v in grouped.items() if len(v) >= min_samples}
    return [s for s in samples if s.identity in keep]


def write_metrics(metrics: Dict[str, EvalMetrics], out_path: str) -> None:
    serializable = {
        name: {
            "eer": m.eer,
            "eer_threshold": m.eer_threshold,
            "far_at_eer": m.far_at_eer,
            "frr_at_eer": m.frr_at_eer,
            "auc": m.auc,
        }
        for name, m in metrics.items()
    }
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(serializable, f, indent=2)


def main() -> None:
    parser = argparse.ArgumentParser(description="Multimodal biometrics demo")
    parser.add_argument("--face-mode", choices=["lfw", "folder"], default="lfw")
    parser.add_argument("--face-dir", type=str, default=None, help="Folder dataset for faces if --face-mode folder")
    parser.add_argument("--fingerprint-dir", type=str, required=True)
    parser.add_argument("--palm-dir", type=str, required=True)
    parser.add_argument("--face-model", type=str, default="facenet", help="facenet or a timm model like resnet50")
    parser.add_argument("--device", type=str, default="cpu")
    parser.add_argument("--max-identities", type=int, default=40)
    parser.add_argument("--max-images-per-identity", type=int, default=6)
    parser.add_argument("--output-dir", type=str, default="results")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    set_seed(args.seed)
    out_dir = Path(args.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # Load datasets
    if args.face_mode == "lfw":
        face_samples = load_lfw_dataset(
            min_faces_per_person=5,
            resize_factor=0.5,
            max_identities=args.max_identities,
            max_images_per_identity=args.max_images_per_identity,
        )
    else:
        if not args.face_dir:
            raise ValueError("--face-dir is required when --face-mode folder")
        face_samples = load_folder_dataset(
            args.face_dir,
            color=True,
            max_identities=args.max_identities,
            max_images_per_identity=args.max_images_per_identity,
        )

    fp_samples = load_folder_dataset(
        args.fingerprint_dir,
        color=False,
        max_identities=args.max_identities,
        max_images_per_identity=args.max_images_per_identity,
    )
    palm_samples = load_folder_dataset(
        args.palm_dir,
        color=False,
        max_identities=args.max_identities,
        max_images_per_identity=args.max_images_per_identity,
    )

    face_samples = filter_identities_with_min_samples(face_samples, 2)
    fp_samples = filter_identities_with_min_samples(fp_samples, 2)
    palm_samples = filter_identities_with_min_samples(palm_samples, 2)

    check_min_images(face_samples, "Face")
    check_min_images(fp_samples, "Fingerprint")
    check_min_images(palm_samples, "Palm")

    # Use only identities common to all three modalities
    common_ids = set(s.identity for s in face_samples) & set(s.identity for s in fp_samples) & set(s.identity for s in palm_samples)
    if len(common_ids) < 2:
        raise RuntimeError(
            "Need at least 2 common identities across all three modalities. "
            "Rename / align your subject folders so the same identity has the same folder name in face, fingerprint, and palm datasets."
        )

    face_samples = [s for s in face_samples if s.identity in common_ids]
    fp_samples = [s for s in fp_samples if s.identity in common_ids]
    palm_samples = [s for s in palm_samples if s.identity in common_ids]

    print(f"Common identities across modalities: {len(common_ids)}")
    print(f"Face samples: {len(face_samples)}")
    print(f"Fingerprint samples: {len(fp_samples)}")
    print(f"Palm samples: {len(palm_samples)}")

    # Extract features
    face_feats = extract_face_embeddings(face_samples, model_name=args.face_model, device=args.device)
    fp_minutiae, fp_cnn = extract_fingerprint_features(fp_samples, device=args.device)
    palm_gabor, palm_cnn = extract_palm_features(palm_samples, device=args.device)

    # Score all aligned multimodal pairs
    scores_df = build_score_table(face_samples, face_feats, fp_samples, fp_minutiae, fp_cnn, palm_samples, palm_gabor, palm_cnn)
    scores_csv = out_dir / "scores.csv"
    scores_df.to_csv(scores_csv, index=False)

    # Fusion and evaluation
    eval_df, metrics, fusion_model = train_fusion_models(scores_df, random_state=args.seed)
    eval_csv = out_dir / "eval_scores.csv"
    eval_df.to_csv(eval_csv, index=False)
    write_metrics(metrics, str(out_dir / "metrics.json"))

    # ROC plots
    y = eval_df["label"].values
    for col in ["face_score", "fp_score", "palm_score", "weighted_fusion", "logistic_fusion"]:
        plot_roc(y, eval_df[col].values, title=f"ROC - {col}", out_path=str(out_dir / f"roc_{col}.png"))

    # Save fusion model coefficients for interpretability
    coef_info = {
        "intercept": fusion_model.intercept_.tolist(),
        "coefficients": {
            "face_score": float(fusion_model.coef_[0][0]),
            "fp_score": float(fusion_model.coef_[0][1]),
            "palm_score": float(fusion_model.coef_[0][2]),
        },
    }
    with open(out_dir / "fusion_model.json", "w", encoding="utf-8") as f:
        json.dump(coef_info, f, indent=2)

    print("\n=== Evaluation summary ===")
    for name, m in metrics.items():
        print(f"{name:16s} | EER={m.eer:.4f} | AUC={m.auc:.4f} | threshold={m.eer_threshold:.4f}")

    print(f"\nSaved outputs to: {out_dir.resolve()}")


if __name__ == "__main__":
    main()
